# FlashAttention: Exact Attention at the Speed of Memory

The runnable code of [FlashAttention: Exact Attention at the Speed of Memory](https://residual-stream.lvtan71.workers.dev/posts/flashattention-explained/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/flashattention-explained.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.
Timings and memory depend on the machine; the comparison between the two methods is what carries over.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "flashattention-explained"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Where standard attention's time goes

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/flashattention-explained/#where-standard-attentions-time-goes)

In [ ]:
# Memory traffic of one attention head's forward pass, in 16-bit numbers
# (2 bytes each), for sequence length n and head size d.
#
# Standard attention runs three separate kernels, and each one reads its
# inputs from GPU memory (HBM) and writes its result back:
#   S = Q K^T       reads Q, K (2nd), writes S (n^2)
#   P = softmax(S)  reads S, writes P (2 n^2)
#   O = P V         reads P, V (n^2 + nd), writes O (nd)
#
# FlashAttention, in the loop order FlashAttention-2 uses, keeps a block of
# B_r query rows in fast on-chip memory and streams K and V past it in
# blocks: Q and O cross HBM once, K and V once per block of query rows, and
# S and P never do. (This charges every re-read of K and V to HBM, a worst
# case: on a real GPU many of them hit the L2 cache.)
H100_SMEM = 228 * 1024                                    # bytes of shared memory per SM ("How a GPU Runs Deep Learning")

def standard(n, d):
    return 2 * (2 * n * d + n * n + 2 * n * n + n * n + n * d + n * d)

def flash(n, d, B_r):
    return 2 * (2 * n * d + (n // B_r) * 2 * n * d)       # Q and O once; K and V once per row block

d = 128
B_r = 128                                                 # 128 query rows of 128: 32 KB in 16 bits; fits with K, V blocks
                                                          # (the score tile and output accumulator live in registers)
print(f"one block of Q, K and V at {B_r} x {d} in 16 bits: {3 * B_r * d * 2 / 1024:.0f} KB of the {H100_SMEM // 1024} KB")
for n in (4096, 32768, 131072):
    s, f = standard(n, d), flash(n, d, B_r)
    print(f"n = {n:>7,}: standard {s / 2**20:9,.0f} MiB, flash {f / 2**20:7,.0f} MiB ({s / f:4.1f}x less); "
          f"the score matrix alone {n * n * 2 / 2**20:9,.0f} MiB per head")

# Bigger on-chip memory means bigger row blocks and fewer passes over K and V.
n = 32768
print(f"n = {n:,}: flash traffic with row blocks of " + ", ".join(
    f"{b}: {flash(n, d, b) / 2**20:,.0f} MiB" for b in (64, 128, 256)))

# What training keeps for the backward pass: standard attention saves P,
# n^2 numbers per head; FlashAttention saves only O and one number per row
# (the softmax's log-sum-exp) and recomputes the blocks of P when needed.
for n in (4096, 32768):
    print(f"n = {n:>6,}: saved for backward per head: standard {n * n * 2 / 2**20:7,.0f} MiB, "
          f"flash {(n * d + n) * 2 / 2**20:5.1f} MiB")

The post printed:

```text
one block of Q, K and V at 128 x 128 in 16 bits: 96 KB of the 228 KB
n =   4,096: standard       132 MiB, flash      66 MiB ( 2.0x less); the score matrix alone        32 MiB per head
n =  32,768: standard     8,224 MiB, flash   4,112 MiB ( 2.0x less); the score matrix alone     2,048 MiB per head
n = 131,072: standard   131,200 MiB, flash  65,600 MiB ( 2.0x less); the score matrix alone    32,768 MiB per head
n = 32,768: flash traffic with row blocks of 64: 8,208 MiB, 128: 4,112 MiB, 256: 2,064 MiB
n =  4,096: saved for backward per head: standard      32 MiB, flash   1.0 MiB
n = 32,768: saved for backward per head: standard   2,048 MiB, flash   8.1 MiB
```

## The online softmax

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/flashattention-explained/#the-online-softmax)

In [ ]:
import numpy as np

# Attention one block of keys at a time, never holding a full row of scores.
# The running state per query row: m, the largest score seen so far; l, the
# sum of exp(score - m) so far; o, the weighted sum of values so far, also
# relative to m. When a new block raises the max from m to m', everything
# accumulated is rescaled by exp(m - m') (Milakov and Gimelshein, 2018).
rng = np.random.default_rng(0)

def attention(Q, K, V):
    S = Q @ K.T / np.sqrt(Q.shape[1])                     # the full n x n score matrix
    P = np.exp(S - S.max(1, keepdims=True))
    return (P / P.sum(1, keepdims=True)) @ V

def tiled_attention(Q, K, V, block):
    n, d = Q.shape
    m = np.full(n, -np.inf)                               # running max per row
    l = np.zeros(n)                                       # running sum of exponentials
    o = np.zeros((n, V.shape[1]))                         # running (unnormalized) output
    for j in range(0, K.shape[0], block):
        S = Q @ K[j:j + block].T / np.sqrt(d)             # only n x block scores at a time
        m_new = np.maximum(m, S.max(1))
        scale = np.exp(m - m_new)                         # how much the old max overstated
        P = np.exp(S - m_new[:, None])
        l = l * scale + P.sum(1)
        o = o * scale[:, None] + P @ V[j:j + block]
        m = m_new
    return o / l[:, None]

# The worked example: one query, four keys, two blocks of two.
q = np.array([[1.0, 0.0]])
k = np.array([[2.0, 0.0], [1.0, 0.0], [4.0, 0.0], [0.0, 0.0]]) * np.sqrt(2)   # scores 2, 1, 4, 0
v = np.array([[1.0], [2.0], [3.0], [4.0]])
s = (q @ k.T / np.sqrt(2))[0]
print("scores", s, "-> softmax", np.round(np.exp(s - s.max()) / np.exp(s - s.max()).sum(), 4))
m, l, o = -np.inf, 0.0, 0.0
for j, blk in enumerate(((0, 1), (2, 3))):
    sb = s[list(blk)]
    m_new = max(m, sb.max())
    scale = np.exp(m - m_new)
    p = np.exp(sb - m_new)
    l, o, m = l * scale + p.sum(), o * scale + (p * v[list(blk), 0]).sum(), m_new
    print(f"after block {j + 1}: max {m:.0f}, sum {l:.4f}, output so far {o:.4f} -> {o / l:.4f}")
print(f"standard attention gives {attention(q, k, v)[0, 0]:.4f}")

for n, d, block in ((1024, 64, 128), (4096, 128, 256)):
    Q, K, V = (rng.normal(size=(n, d)) for _ in range(3))
    err = np.abs(tiled_attention(Q, K, V, block) - attention(Q, K, V)).max()
    print(f"n = {n}, d = {d}, blocks of {block}: largest difference from standard attention {err:.1e}")

The post printed:

```text
scores [2. 1. 4. 0.] -> softmax [0.1125 0.0414 0.831  0.0152]
after block 1: max 2, sum 1.3679, output so far 1.7358 -> 1.2689
after block 2: max 4, sum 1.2034, output so far 3.3082 -> 2.7489
standard attention gives 2.7489
n = 1024, d = 64, blocks of 128: largest difference from standard attention 7.8e-16
n = 4096, d = 128, blocks of 256: largest difference from standard attention 6.8e-16
```

## Measured

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/flashattention-explained/#measured)

In [ ]:
%%writefile step4.py
import resource
import subprocess
import sys
import time
import torch
import torch.nn.functional as F

# Standard attention (score matrix, softmax, weighted sum: three operations)
# against PyTorch's fused scaled_dot_product_attention, which on the CPU
# runs a FlashAttention-style tiled kernel. One head of size 64, 32-bit
# floats. Each measurement runs in a fresh process so its peak memory can
# be read on its own. Run on an otherwise idle machine: under memory
# pressure the standard version's big matrices get swapped, and its times
# become meaningless.
def run(kind, n, d=64):
    torch.manual_seed(0)
    q, k, v = (torch.randn(1, 1, n, d) for _ in range(3))
    def standard():
        s = q @ k.transpose(-1, -2) / d ** 0.5
        return torch.softmax(s, -1) @ v
    def fused():
        return F.scaled_dot_product_attention(q, k, v)
    f = standard if kind == "standard" else fused
    f()
    t0 = time.perf_counter()
    for _ in range(3):
        out = f()
    t = (time.perf_counter() - t0) / 3
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * (1 if sys.platform == "darwin" else 1024) / 2**20   # notebook: bytes on macOS, KiB on Linux
    print(f"{t * 1e3:.1f} {peak:.0f}")

if len(sys.argv) > 1:
    torch.set_num_threads(4)
    run(sys.argv[1], int(sys.argv[2]))
else:
    base = None
    for n in (2048, 8192, 16384):
        row = {}
        for kind in ("standard", "fused"):
            out = subprocess.run([sys.executable, __file__, kind, str(n)], capture_output=True, text=True).stdout.split()
            row[kind] = (float(out[0]), float(out[1]))
        (ts, ms), (tf, mf) = row["standard"], row["fused"]
        print(f"n = {n:>6,}: standard {ts:7.1f} ms, peak {ms:6,.0f} MiB;  fused {tf:7.1f} ms, peak {mf:6,.0f} MiB;  "
              f"the n x n score matrix is {n * n * 4 / 2**20:5,.0f} MiB")
    torch.manual_seed(0)
    a = torch.randn(1, 1, 4096, 64); b = torch.randn(1, 1, 4096, 64); c = torch.randn(1, 1, 4096, 64)
    ref = torch.softmax(a @ b.transpose(-1, -2) / 8, -1) @ c
    print(f"largest difference between the two at n = 4,096: {(ref - F.scaled_dot_product_attention(a, b, c)).abs().max():.1e}")

In [ ]:
%run step4.py

The post printed:

```text
n =  2,048: standard     3.9 ms, peak    228 MiB;  fused     1.1 ms, peak    197 MiB;  the n x n score matrix is    16 MiB
n =  8,192: standard    44.4 ms, peak    716 MiB;  fused    14.7 ms, peak    206 MiB;  the n x n score matrix is   256 MiB
n = 16,384: standard   175.6 ms, peak  2,263 MiB;  fused    57.7 ms, peak    215 MiB;  the n x n score matrix is 1,024 MiB
largest difference between the two at n = 4,096: 6.6e-07
```